In [1]:
!pip3 install gurobipy

## 📌 Problema del flusso a costo minimo
Dato un grafo $G = (V, E)$, con capacita' su gli archi pari ad $h_{ij}$ per ogni arco $(i, j) \in E$, domanda (positiva o negativa) pari a $b_i$ per ogni nodo $i \in V$, e costo unitario del flusso $c_{ij}$ per ogni arco $(i, j) \in E$ il problema del flusso a costo minimo consiste nel trovare un flusso che rispetti tutte le domande a costo minimo.

### 1. Generazione dei dati

Cominciamo con il generare in modo casuale un grafo $G$ (per facilitare la riproduzione dei risultati e' consigliabile fissare il seed del generatore di numeri casuali utilizzati, ad esempio usando np.random.seed) con almeno 150 nodi.

Generiamo inoltre le capacita' per ogni arco, nel range tra 1 e 10, e le domande per ogni nodo, assicurandoci che almeno un nodo abbia domanda positiva (e che il totale della domanda positiva sia maggiore di 15) ed almeno un altro nodo abbia domanda negativa (e che il totale della domanda negativa sia pari a quello della domanda positiva). Inoltre generiamo i costi per ogni arco nel grafo come numeri tra 1 ed 10.

In [25]:
import numpy as np

# Fisso il seed per avere risultati riproducibili
np.random.seed(42)

# Numero di nodi
n_nodes = 50

# Generazione del Grafo
adj_mat = np.random.rand(n_nodes, n_nodes)  # Genero una matrice 150x150 di numeri casuali tra 0 e 1
adj_mat = (adj_mat < 0.5).astype(int)      # Rendo il grafico sparso: tengo solo gli archi con prababilità < 0.05
np.fill_diagonal(adj_mat, 0)                # Rimuovo i ceppi (nodi collegati a se stessi) 


# Generazione costi e capacità
capacities = np.random.randint (1, 11, size=(n_nodes, n_nodes)) * adj_mat   # Capacità
costi = np.random.randint (1, 11, size=(n_nodes, n_nodes)) * adj_mat         # Costo

# Definisco le domade/offerte (b_i):   Sorgente: b_i < 0         Destinazione:b_i > 0
demands = np.zeros(n_nodes)
demands[0:5] = -20  # Sorgenti (offere)
demands[-5:] = 20   # Destinazione (domanda)

# Verifica bilancio totale (deve essere 0)
print(f"Bilancio totale domande: {np.sum(demands)}")

Bilancio totale domande: 0.0


### 2. Definizione del Modello, Variabili e Vincoli

In [26]:
import gurobipy as gp
from gurobipy import GRB

# Inizializzazione modello
model = gp.Model("Flusso_Costo_Minimo")
model.Params.LogToConsole = 0               # Toglie tutte le scritte su Gurobi

# Identificazione archi esistenti
var_idxs = np.argwhere(adj_mat > 0)         # Trovo le coordinate (i, j) per gli archi creati
arcs = [(i, j) for i, j in var_idxs]        # Trasformo le coordinate in una lista di tuple

# Definizione variabili decisionali
x = model.addVars(
    arcs, 
    lb=0,                                      # lb = 0 (il flusso non può essere negativo)
    ub=[capacities[i, j] for i, j in arcs],    # ub = capacità massima specifica di quell'arco
    name="flusso"
)

# Vincoli di conservazione del flusso
for v in range(n_nodes):
    flusso_entrate = gp.quicksum(x[i, v] for i in range(n_nodes) if adj_mat[i, v] == 1)         # Vincolo Entarte
    flusso_uscite = gp.quicksum(x[v, j] for j in range(n_nodes) if adj_mat[v, j] == 1)          # Vincolo Uscite
    model.addConstr(flusso_entrate - flusso_uscite == demands[v], name=f"bilancio_{v}")         # Entrate - Uscite = Domanda

# Funzione obiettivo: Minimizzazione costi totali
costo_totale = gp.quicksum(costi[i, j] * x[i, j] for i, j in arcs)
model.setObjective(costo_totale, GRB.MINIMIZE)

# Ottimizzazione
model.optimize()

Set parameter LogToConsole to value 0


### 3. Analisi dei risultati

In [27]:
if model.Status == GRB.OPTIMAL:
    print(f"\nCosto minimo totale per soddisfare la rete: {model.ObjVal} €\n")
    
    print("Dettaglio di alcuni archi utilizzati (flusso > 0):")
    # Stampiamo solo i primi 10 archi utilizzati per non intasare l'output
    count = 0
    for i, j in arcs:
        if x[i, j].X > 0:
            print(f"Arco ({i} -> {j}): trasporta {x[i, j].X} unità (costo unitario: {costi[i, j]} €)")
            count += 1
            if count >= 10:
                print("...")
                break
else:
    print("\nIl modello non ha trovato una soluzione ottima. Verifica i dati di input.")


Costo minimo totale per soddisfare la rete: 430.0 €

Dettaglio di alcuni archi utilizzati (flusso > 0):
Arco (0 -> 18): trasporta 6.0 unità (costo unitario: 3 €)
Arco (0 -> 26): trasporta 9.0 unità (costo unitario: 2 €)
Arco (0 -> 46): trasporta 5.0 unità (costo unitario: 2 €)
Arco (1 -> 10): trasporta 6.0 unità (costo unitario: 2 €)
Arco (1 -> 13): trasporta 2.0 unità (costo unitario: 1 €)
Arco (1 -> 14): trasporta 1.0 unità (costo unitario: 2 €)
Arco (1 -> 21): trasporta 2.0 unità (costo unitario: 3 €)
Arco (1 -> 48): trasporta 9.0 unità (costo unitario: 3 €)
Arco (2 -> 17): trasporta 7.0 unità (costo unitario: 1 €)
Arco (2 -> 22): trasporta 2.0 unità (costo unitario: 3 €)
...


# Esercizio Calzature



Un industria calzaturiera produce 3 diversi modelli di scarpe (A, B e C) in 5 stabilimenti dislocati nel territorio. Le calzature vengono vendute al prezzo di 30, 55 e 40 euro rispettivamente. Dovendo decidere il piano produttivo per il mese seguente, il gruppo industriale deve decidere quali materie prime acquistare e come gestire la produzione per massimizzare il proprio profitto.

Per l’acquisto di materie prime, l’azienda ha a disposizione 1000 euro e puo’ acquistare pelle al prezzo di 4.5 euro/mq, lacci al prezzo di 73 cent/mt e suole al prezzo di 7 euro/unita’.

In un mese gli stabilimenti posso produrre le seguenti quantita di scapre (in paia):

| Modello      | Stabilimento 1 | Stabilimento 2 | Stabilimento 3 | Stabilimento 4 | Stabilimento 5 |
| ----------- | ----------- | ----------- | ----------- | ----------- | ----------- |
| A | 4 | 12 | 3 | 9 | 7 |
| B | 2 | 7 | 3 | 14 | 5 |
| C | 8 | 3 | 5 | 2 | 9 |


La produzione di una calzatura richiede una suola per tutti I modelli di scapre, ed in oltre: 1mq di pelle e 50cm di lacci per il modello A, 0.7mq di pelle e 45cm di lacci per il modello B, 1.35mq di pelle e 90cm di lacci per il modello C.

Considerando che le materie prime acquistate posso essere distribuite agli stabilimenti produttivi in tagli da 10mq di pelle, 5mt di lacci e blocchi di 3 paia di suole. Si determini l’approvigionamento del gruppo industriale, la distribuzione delle materie prime e la produzione in paia dei singoli stabilimenti in modo tale da massimizzare il profitto del gruppo, asssumendo che tutta la produzione venga venduta e sapendo ulteriormente che:

1. Devono essere prodotte almeno 5 paia di scarpe di tipo A

2. La produzione in paia dello stabilimento 3 deve essere pari a quella dello stabilimento 1

3. Lo stabilimento 4 ha gia’ ricevuto una commessa per 2 paia di scarpe di tipo B

###  1. Definizione dati e variabili

In [28]:
import gurobipy as gp
from gurobipy import GRB

# Creazione del modello
model = gp.Model("Produzione_Calzature")
model.Params.LogToConsole = 0               # Toglie tutte le scritte su Gurobi

# Liste per indici
modelli = ['A', 'B', 'C']
stabilimenti = [1, 2, 3, 4, 5]
materie = ['Pelle', 'Lacci', 'Suole']

# Prezzi di vendita
prezzi = {'A': 30, 'B': 55, 'C': 40}

# Requisiti di materie prime per ogni paio di scarpe
req = {
    'A': {'Pelle': 1.0,  'Lacci': 0.50, 'Suole': 1},
    'B': {'Pelle': 0.7,  'Lacci': 0.45, 'Suole': 1},
    'C': {'Pelle': 1.35, 'Lacci': 0.90, 'Suole': 1}
}

# Capacità produttiva massima per [Modello, Stabilimento]
max_p = {
    'A': {1: 4, 2: 12, 3: 3, 4: 9, 5: 7},
    'B': {1: 2, 2: 7, 3: 3, 4: 14, 5: 5},
    'C': {1: 8, 2: 3, 3: 5, 4: 2, 5: 9}
}

# Dati acquisto blocchi materie prime
costo_b = {'Pelle': 45, 'Lacci': 3.65, 'Suole': 21}         # Costo per singolo blocco/taglio
qta_b = {'Pelle': 10, 'Lacci': 5, 'Suole': 3}               # Quantità contenuta in ogni blocco

# Variabili decisionali
x = model.addVars(modelli, stabilimenti, vtype=GRB.INTEGER, name="x") # Produzione
d = model.addVars(materie, stabilimenti, vtype=GRB.INTEGER, name="d") # Blocchi materie

Set parameter LogToConsole to value 0


### 2. Vincoli e Obiettivo

In [29]:
# Vincolo budget totale (1000 euro)
costo_acquisto = gp.quicksum(costo_b[m] * d[m, s] for m in materie for s in stabilimenti)
model.addConstr(costo_acquisto <= 1000, name="budget")

# Vincolo bilancio materie prime per stabilimento
for s in stabilimenti:
    for mat in materie:
        consumo = gp.quicksum(req[mod][mat] * x[mod, s] for mod in modelli)                     # Somma del consumo di ogni modello 
        model.addConstr(consumo <= qta_b[mat] * d[mat, s], name=f"fabbisogno_{mat}_{s}")        # Deve essere <= alla quantita totale

# Capacità produttiva
for m in modelli:
    for s in stabilimenti:
        model.addConstr(x[m, s] <= max_p[m][s])

# Vincoli traccia
model.addConstr(                    # Minimo 5 paia A
    x.sum('A', '*') >= 5,
      name = "min_A"
    )

model.addConstr(                    # Produzione Stab 3 = Stab 1
    x.sum('*', 3) == x.sum('*', 1),
    name = "eq_1_3"
    ) 

model.addConstr(                    # Commessa fissa B4
    x['B', 4] >= 2,
    name = "comessa_B4"
    ) 

# Obiettivo: Profitto (Ricavi - Costi)
ricavi = gp.quicksum(prezzi[m] * x[m, s] for m in modelli for s in stabilimenti)
costi = gp.quicksum(costo_b[m] * d[m, s] for m in materie for s in stabilimenti)
model.setObjective(ricavi - costi, GRB.MAXIMIZE)

# Risoluzione
model.optimize()

### 3. Analisi risultati

In [30]:
if model.Status == GRB.OPTIMAL:
    print(f"Profitto Totale Ottimizzato: {model.ObjVal} €\n")
    
    for s in stabilimenti:
        # Recupero i valori di produzione diversi da zero
        prod_effettiva = {m: x[m, s].X for m in modelli if x[m, s].X > 0}
        if prod_effettiva:
            print(f"Stabilimento {s} produce: {prod_effettiva}")
            
    print("\nDistribuzione Materie Prime (Blocchi):")
    for s in stabilimenti:
        blocchi_s = {mat: int(d[mat, s].X) for mat in materie if d[mat, s].X > 0}
        print(f"Stabilimento {s} riceve: {blocchi_s}")

Profitto Totale Ottimizzato: 2430.55 €

Stabilimento 1 produce: {'A': 3.0, 'B': 2.0, 'C': 4.0}
Stabilimento 2 produce: {'A': 11.0, 'B': 7.0, 'C': 3.0}
Stabilimento 3 produce: {'A': 1.0, 'B': 3.0, 'C': 5.0}
Stabilimento 4 produce: {'A': 7.0, 'B': 14.0, 'C': 2.0}
Stabilimento 5 produce: {'A': 4.0, 'B': 5.0, 'C': 9.0}

Distribuzione Materie Prime (Blocchi):
Stabilimento 1 riceve: {'Pelle': 1, 'Lacci': 2, 'Suole': 3}
Stabilimento 2 riceve: {'Pelle': 2, 'Lacci': 3, 'Suole': 7}
Stabilimento 3 riceve: {'Pelle': 1, 'Lacci': 2, 'Suole': 3}
Stabilimento 4 riceve: {'Pelle': 2, 'Lacci': 3, 'Suole': 8}
Stabilimento 5 riceve: {'Pelle': 2, 'Lacci': 3, 'Suole': 6}
